# Text-to-Speech Demo

## 1. Setup

In [ ]:
# Install missing packages
!pip install -q -U omnivoice pydub gdown

In [ ]:
import os
import io
import json
import wave
import torch
import gdown
import shutil
import zipfile
import numpy as np
from base64 import b64decode
from pydub import AudioSegment
from omnivoice import OmniVoice
from google.colab.output import eval_js
from IPython.display import Javascript, Audio, display

In [ ]:
_RECORD_JS = """
async function record(ms, prompt) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });

  const div = document.createElement('div');
  div.innerHTML = `<p>Click, then <b>${prompt}</b> (auto-stops after ${ms / 1000}s)</p><button>🎤 Start</button>`;
  document.body.appendChild(div);
  const btn = div.querySelector('button');
  await new Promise(resolve => { btn.onclick = resolve; });
  btn.textContent = `● Recording... (${ms / 1000}s)`;

  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  setTimeout(() => recorder.stop(), ms);
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  div.remove();

  const reader = new FileReader();
  reader.readAsDataURL(new Blob(chunks));
  return new Promise(resolve => { reader.onloadend = () => resolve(reader.result); });
}
"""

def _to_waveform(segment):
    segment = segment.set_channels(1)
    waveform = np.array(segment.get_array_of_samples()).astype(np.float32)
    waveform /= 1 << (8 * segment.sample_width - 1)
    return waveform, segment.frame_rate

def record_audio(seconds=8, prompt="speak now"):
    """Record from the browser mic, returns (waveform float32 numpy, sample_rate)."""
    display(Javascript(_RECORD_JS))
    data_url = eval_js(f"record({seconds * 1000}, {json.dumps(prompt)})")
    raw = b64decode(data_url.split(",", 1)[1])
    return _to_waveform(AudioSegment.from_file(io.BytesIO(raw)))

my_recordings = {}
my_transcripts = {}

def save_recording(name, waveform, sr, text=None):
    """Keep a recording (and its known text, if given) in memory for this session - not written to disk or Drive, gone on runtime reset."""
    my_recordings[name] = (waveform, sr)
    if text is not None:
        my_transcripts[name] = text

def clear_recordings():
    """Delete every in-session recording (and its text) from memory - no undo."""
    my_recordings.clear()
    my_transcripts.clear()

def download_recordings():
    """Zip every recording + a transcripts.json (same format as the common bank) and trigger a browser download to your PC."""
    from google.colab import files

    zip_path = "/content/my_recordings.zip"
    with zipfile.ZipFile(zip_path, "w") as zf:
        for name, (waveform, sr) in my_recordings.items():
            pcm16 = np.clip(waveform * 32768, -32768, 32767).astype(np.int16)
            wav_path = f"/content/{name}.wav"
            with wave.open(wav_path, "wb") as wav_file:
                wav_file.setnchannels(1)
                wav_file.setsampwidth(2)
                wav_file.setframerate(sr)
                wav_file.writeframes(pcm16.tobytes())
            zf.write(wav_path, arcname=f"{name}.wav")
        zf.writestr("transcripts.json", json.dumps(my_transcripts, ensure_ascii=False, indent=2))
    files.download(zip_path)

def load_sample(name):
    """Load a clip by name - checks your in-session recordings (Section 3) first, then the common bank (Section 2)."""
    if name in my_recordings:
        return my_recordings[name]
    path = f"{AUDIO_SAMPLES_DIR}/{name}"
    if os.path.exists(path):
        return _to_waveform(AudioSegment.from_file(path))
    raise FileNotFoundError(f"'{name}' not found in your recordings or {AUDIO_SAMPLES_DIR}")

## 2. Load audio samples

In [ ]:
# audio samples: reusing the SAME shared zip as the speech-to-text demo -
# its clips + transcripts.json (audio + known text) work equally well as
# cloning references here, no need for a separate TTS-only bank
AUDIO_SAMPLES_DIR = "/content/audio_samples"
AUDIO_SAMPLES_FOLDER_ID = "1YgInfep4vnRA1pX4-7h8e4G4AxqzPLg6"

In [ ]:
shutil.rmtree(AUDIO_SAMPLES_DIR, ignore_errors=True)
os.makedirs(AUDIO_SAMPLES_DIR, exist_ok=True)

try:

    # Listing the folder is one lightweight request (not a download); find
    # whichever .zip is in there right now instead of hardcoding a file id.
    listing = gdown.download_folder(id=AUDIO_SAMPLES_FOLDER_ID, skip_download=True, use_cookies=False)
    zip_entries = [f for f in listing if f.path.endswith(".zip")]
    if len(zip_entries) != 1:
        print(f"Expected exactly one .zip in the folder, found {len(zip_entries)}: {[f.path for f in zip_entries]} - using the first one.")
    zip_path = gdown.download(id=zip_entries[0].id, output="/content/audio_samples.zip", quiet=False)

    with zipfile.ZipFile(zip_path) as zf:
        # Flatten structure - extract every file to AUDIO_SAMPLES_DIR directly
        for member in zf.namelist():
            if member.endswith("/"):
                continue
            with zf.open(member) as src, open(f"{AUDIO_SAMPLES_DIR}/{os.path.basename(member)}", "wb") as dst:
                dst.write(src.read())
except Exception as e:
    print(f"Couldn't download the common sample bank ({e!r}) — you can still use Section 3 to record your own samples.")

audio_samples = sorted(f for f in os.listdir(AUDIO_SAMPLES_DIR) if f != "transcripts.json")

transcripts_path = f"{AUDIO_SAMPLES_DIR}/transcripts.json"
if os.path.exists(transcripts_path):
    with open(transcripts_path) as f:
        transcripts = json.load(f)
else:
    transcripts = {}

print(f"\n{audio_samples}")
print(f"\n{len(audio_samples)} audio sample(s) loaded.")
print(f"\n{len(transcripts)} reference transcript(s) loaded.")

In [ ]:
# Preview all samples.
for name in audio_samples:
    print(f"\n{name}\n{transcripts[name]}")
    display(Audio(f"{AUDIO_SAMPLES_DIR}/{name}"))

## 3. Create new samples (yours only)
- set sample name
- set text - exactly what you will say, needed as `ref_text` to clone your voice later
- run the cell, it will fail until you allow mic and run it again
- click start and speak, the duration of the recording is fixed by the function argument

In [ ]:
# sample_name = "my_voice_01"
# text = "The quick brown fox jumps over the lazy dog. NLP is great!"

# waveform, sr = record_audio(8, prompt=f'read: "{text}"')
# save_recording(sample_name, waveform, sr, text=text)
# display(Audio(waveform, rate=sr))

Preview everything you've recorded so far.

In [ ]:
for name, (waveform, sr) in my_recordings.items():
    print(f"{name}: {my_transcripts.get(name, '(no reference text saved)')}")
    display(Audio(waveform, rate=sr))

Download recordings if you want to keep them after the runtime resets.

In [ ]:
# download_recordings()

Delete current recordings to start over.

In [ ]:
# clear_recordings()

## 4. Load models

We use [`k2-fsa/OmniVoice`](https://huggingface.co/k2-fsa/OmniVoice), a 0.6B multilingual TTS model supporting both **voice design** (describe a voice via attributes) and **voice cloning** (clone a voice from a short reference clip).

In [ ]:
MODEL_ID = "k2-fsa/OmniVoice"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device.startswith("cuda") else torch.float32
print(device)

model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)

## 5. Synthesize: pick text & voice

`synthesize(text, name, instruct=None, ref_sample=None, ref_text=None, language=None)` ties everything together:

- `text`: what to say.
- `name`: saves the output as `outputs/<name>.wav` and plays it back.
- `instruct`: voice design attributes (e.g. `"female, elderly, british accent"`) - see the vocabulary below. Use this OR `ref_sample`, not both.
- `ref_sample`: a filename from the common bank (Section 2) or a name you gave in `save_recording()` (Section 3), to clone that voice instead of describing one.
- `ref_text`: what's said in `ref_sample`. Auto-filled from `transcripts.json` or `my_transcripts` (populated by `save_recording(..., text=...)`) - pass explicitly only to override.
- `language`: a code (`"sk"`) or full name (`"Slovak"`); omitted/invalid falls back to language-agnostic mode (inferred from text) with just a warning.

In [ ]:
def synthesize(text, name, instruct=None, ref_sample=None, ref_text=None, language=None):
    kwargs = {"text": text, "language": language}
    if ref_sample:
        waveform, sr = load_sample(ref_sample)
        kwargs["ref_audio"] = (waveform, sr)
        kwargs["ref_text"] = ref_text if ref_text is not None else (my_transcripts.get(ref_sample) or transcripts.get(ref_sample))
    else:
        kwargs["instruct"] = instruct

    audio = model.generate(**kwargs)
    if isinstance(audio, list):
        audio = audio[0]

    os.makedirs("outputs", exist_ok=True)
    path = f"outputs/{name}.wav"
    pcm16 = np.clip(audio * 32768, -32768, 32767).astype(np.int16)
    with wave.open(path, "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(24000)
        wav_file.writeframes(pcm16.tobytes())
    display(Audio(path))

### Voice design

`instruct`: comma-separated, at most one item per category (English only):
- gender: male, female
- age: child, teenager, young adult, middle-aged, elderly
- pitch: very low pitch, low pitch, moderate pitch, high pitch, very high pitch
- style: whisper
- accent: american, british, australian, canadian, indian, japanese, korean, portuguese, russian, chinese accent

e.g. `"female, elderly, low pitch, british accent"`

In [ ]:
synthesize("Hello NLP Summer School of 2026. Welcome in Kinit.", "en_female_welcome", instruct="female")
synthesize("Ahoj, letná škola NLP 2026. Vitajte v Kinite.", "sk_male_welcome", instruct="male", language="sk")
synthesize("Ahoj, letná škola NLP 2026. Vitajte v Kinyte.", "sk_male_welcome", instruct="male", language="sk")

### Voice cloning

Instead of describing a voice with `instruct`, clone one from a reference clip in the common bank (Section 2).

In [ ]:
synthesize(
    "Now I can speak using someone else's voice, thanks to voice cloning!",
    "cloned_sample_voice",
    ref_sample="en_librispeech_01.wav",
)

### Clone your own voice

Uses the recording from Section 3.

In [ ]:
synthesize(
    "Surprise, this is my voice, cloned by an AI, saying something I never actually said!",
    "my_voice_clone",
    ref_sample="my_voice_01",
)